In [ ]:
pip install groq python-dotenv

In [ ]:
import json
import os
from dotenv import find_dotenv, load_dotenv
import joblib
import pandas as pd
from groq import Groq

# 1. Chargement de la clé API depuis .env avec override
load_dotenv(find_dotenv(), override=True)

# On récupère la clé de manière sécurisée (sans l'afficher)
groq_api_key = os.getenv("GROQ_API_KEY")

# Initialisation du client avec la clé récupérée
client = Groq(api_key=groq_api_key)

# 2. Chargement du modèle ML
model_data = joblib.load('../models/mro_risk_model.pkl')
model = model_data['model']
features = model_data['features']

print(" Modèle et Client Groq initialisés avec succès !")

# 3. Fonction d'inférence ML
def predict_late_risk(scenario_dict):
    df_single = pd.DataFrame([scenario_dict])
    df_encoded = pd.get_dummies(df_single)
    df_aligned = df_encoded.reindex(columns=features, fill_value=0)
    
    # Calcul de la probabilité de retard
    risk_proba = model.predict_proba(df_aligned)[0][1]
    return risk_proba

# 4. Agent LLM Groq (MRO Decision Engine)
def run_mro_llm_agent(scenario, risk_score):
    system_prompt = """
    Tu es un Expert Senior en Supply Chain Aéronautique et MRO (Maintenance, Repair, and Overhaul) pour une compagnie aérienne (ex: Royal Air Maroc).
    Ton rôle est d'analyser le score de risque produit par un modèle ML et de fournir un plan d'action préventif clair et structuré pour éviter une immobilisation d'avion (AOG - Aircraft On Ground).

    Structure ta réponse ainsi :
    1. **Diagnostic du Risque** (Explication métier du score ML)
    2. **Impact Opérationnel MRO** (Risque AOG / Maintenance)
    3. **Plan d'Action Immédiat** (2 à 3 mesures logistiques concrètes)
    """

    user_prompt = f"""
    Alerte Commande Pièce Aéronautique :
    - Données Logistiques : {json.dumps(scenario, indent=2)}
    - Risque de Retard Détecté par le ML : {risk_score * 100:.1f}%

    Rédige ton analyse et tes recommandations métiers.
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b", 
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
    )

    return response.choices[0].message.content

In [ ]:
# Lister les modèles disponibles sur votre compte Groq
models_list = client.models.list()

print("--- Modèles actuellement disponibles sur votre compte Groq ---")
for m in models_list.data:
    print(f"- {m.id}")

In [ ]:
# 5. TEST COMPLET DU PIPELINE
sample_order = {
    'ordered_qty': 50,
    'promised_lead_time': 90,  # Délais long (90 jours)
    'qty_fill_rate': 0.70       # Fournisseur qui ne livre pas à 100%
}

# Inférence ML
score = predict_late_risk(sample_order)
print(f"Probabilité de retard (ML) : {score * 100:.2f}%\n")

# Génération par l'Agent LLM
recommendation = run_mro_llm_agent(sample_order, score)
print("=== RECOMMANDATION DE L'AGENT IA (GROQ) ===")
print(recommendation)